# Гео-канал

У запроса есть `search_location_id` (числовой ID зоны, 2782 уникальных значения), у объявления - `item_location_id` + реальные `item_latitude`/`item_longitude`. Разберёмся, как это лучше использовать.

In [2]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd

from eval import make_val_split, recall_at_50
from geo import build_geo_tree, build_loc_centroid, haversine

train = pd.read_parquet("../data/train.parquet")
train_part, val_queries, val_relevant = make_val_split(train, val_size=0.1, random_state=42)
print("train_part:", len(train_part), "val_queries:", len(val_queries))

train_part: 448040 val_queries: 35447


## Локация не бесполезна

`search_location_id == item_location_id` совпадает точно только в **83.1%** случаев (много выездных/удалённых услуг) - НЕ хардфильтр, отдельный сигнал.

In [3]:
(train["search_location_id"] == train["item_location_id"]).mean()

np.float64(0.8310195650557696)

## Локация - это зона, не точка

`location_id` - это район/город, а не точка, и точное совпадение ID теряет информацию о том, НАСКОЛЬКО далеко объявление, если ID не совпал.

In [5]:
items = train.drop_duplicates("item_id").copy()
items["item_latitude"] = items["item_latitude"].astype(float)
items["item_longitude"] = items["item_longitude"].astype(float)

coord_std = items.groupby("item_location_id")[["item_latitude", "item_longitude"]].apply(lambda g: g.std().max())
print("медианный std координат внутри location_id в градусах):", coord_std.median())
print("в км:", coord_std.median() * 111)

медианный std координат внутри location_id в градусах): 0.02004449638661928
в км: 2.22493909891474


## Дистанция вместо ID: у запроса нет своих координат

Запрос даёт только `search_location_id`.\
 Аппроксимируем координаты зоны как **медиану** координат объявлений в `train_part` с тем же `item_location_id` - центроид зоны, без утечки из val.

In [7]:
loc_centroid = build_loc_centroid(train_part)
print("центроидов:", len(loc_centroid))
coverage = val_queries["search_location_id"].isin(loc_centroid.index).mean()
print("доля val_queries с центроидом:", coverage)

центроидов: 2589
доля val_queries с центроидом: 0.8757017519113042


Проверим сигнал напрямую: дистанция до истинного item должна быть намного меньше, чем до случайного.

In [8]:
sample = train.sample(20000, random_state=1).reset_index(drop=True)
sample["item_latitude"] = sample["item_latitude"].astype(float)
sample["item_longitude"] = sample["item_longitude"].astype(float)
centroid = loc_centroid.reindex(sample["search_location_id"]).reset_index(drop=True)
mask = centroid["item_latitude"].notna()
sample, centroid = sample[mask].reset_index(drop=True), centroid[mask].reset_index(drop=True)

dist_true = haversine(centroid["item_latitude"], centroid["item_longitude"], sample["item_latitude"], sample["item_longitude"])
random_items = train.sample(len(sample), random_state=2)[["item_latitude", "item_longitude"]].astype(float).reset_index(drop=True)
dist_random = haversine(centroid["item_latitude"], centroid["item_longitude"], random_items["item_latitude"], random_items["item_longitude"])

print("медиана дистанции до истинного item (км):", dist_true.median())
print("медиана дистанции до случайного item (км):", dist_random.median())
print("доля истинных item в радиусе 5км:", (dist_true <= 5).mean())
print("доля случайных item в радиусе 5км:", (dist_random <= 5).mean())

медиана дистанции до истинного item (км): 2.881070885921557
медиана дистанции до случайного item (км): 1211.7358605718555
доля истинных item в радиусе 5км: 0.662908744180829
доля случайных item в радиусе 5км: 0.006842784227943238


## top50 ближайших по BallTree (haversine)

In [9]:
geo_tree, geo_item_ids = build_geo_tree(train.drop_duplicates("item_id")[["item_id", "item_latitude", "item_longitude"]])

query_coords = loc_centroid.reindex(val_queries["search_location_id"])[["item_latitude", "item_longitude"]].to_numpy()
valid_mask = ~np.isnan(query_coords).any(axis=1)
_, indices = geo_tree.query(np.radians(query_coords[valid_mask]), k=50)

geo_dist_predictions = {}
valid_context_ids = val_queries["context_id"].to_numpy()[valid_mask]
for cid, row in zip(valid_context_ids, indices):
    geo_dist_predictions[cid] = [geo_item_ids[i] for i in row]
for cid in val_queries["context_id"][~valid_mask]:
    geo_dist_predictions[cid] = []

geo_dist_recall = recall_at_50(geo_dist_predictions, val_relevant)
print(f"Гео (top-50 по дистанции) Recall@50: {geo_dist_recall:.4f}")

Гео (top-50 по дистанции) Recall@50: 0.0933


Для сравнения - более простой вариант (exact location_id match + сортировка по популярности в train_part, без реальных координат) даёт **0.0750** - дистанция по факту точнее